# YOLOV5N @ 640px — NSFW object detection

Trains **yolov5n** on the `erotica-detection` dataset (Roboflow **v6**) at **640px** for 150 epochs (early stopping, patience 30), then evaluates on the **test** split with `val.py --task test` at the same resolution and benchmarks inference speed. Structure is shared across the four YOLOv5 base notebooks (n/s x 320/640) so the only variables are model scale and image size.

In [ ]:
!nvidia-smi


In [ ]:
# Roboflow API key from Colab secrets
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")


In [ ]:
# Go to /content and clone YOLOv5
%cd /content
!git clone https://github.com/ultralytics/yolov5
%cd yolov5


In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip uninstall wandb -qy  # deprecated dependency
import torch
print('Setup complete. Using torch %s %s' % (
    torch.__version__,
    torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))


In [ ]:
# Download the dataset (Roboflow v6, YOLOv5 format)
%cd /content/yolov5
!pip install -q roboflow==1.1.48

from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
workspace = rf.workspace("renaire-odarve")
project   = workspace.project("erotica-detection")
version   = project.version(6)
dataset   = version.download("yolov5")


In [ ]:
%%time
# Train yolov5n at 640px
%cd /content/yolov5/
!python train.py \
  --img 640 \
  --batch 64 \
  --epochs 150 \
  --data {dataset.location}/data.yaml \
  --weights yolov5n.pt \
  --name yolov5n_640_results \
  --patience 30 \
  --device 0 \
  --cache


In [ ]:
# Make data.yaml paths absolute so val.py finds the splits
import yaml, os

data_yaml_path = f"{dataset.location}/data.yaml"
with open(data_yaml_path) as f:
    data = yaml.safe_load(f)

data["train"] = os.path.abspath(f"{dataset.location}/train/images")
data["val"]   = os.path.abspath(f"{dataset.location}/valid/images")
data["test"]  = os.path.abspath(f"{dataset.location}/test/images")

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data, f)
print("data.yaml paths updated:", data)


In [ ]:
# Evaluate on the TEST split at 640px
%cd /content/yolov5
!python val.py \
  --weights runs/train/yolov5n_640_results/weights/best.pt \
  --data {dataset.location}/data.yaml \
  --task test \
  --img 640 \
  --batch 64 \
  --verbose


In [ ]:
# Inference-speed benchmark on the test images at 640px
%cd /content/yolov5
!python detect.py \
  --weights runs/train/yolov5n_640_results/weights/best.pt \
  --source {dataset.location}/test/images \
  --img 640 \
  --conf 0.25 --iou 0.45 --device 0 --nosave


In [ ]:
# Training curves
from IPython.display import Image
Image(filename='/content/yolov5/runs/train/yolov5n_640_results/results.png', width=1000)


In [ ]:
# Zip and download the run artifacts
!zip -r yolov5n_640_results.zip /content/yolov5/runs/train/yolov5n_640_results/

from google.colab import files
files.download('yolov5n_640_results.zip')
